# ⚡ LegacyNode — Google Colab GPU Cloud Runner

**Fallback runner for Google Colab when Kaggle is unavailable.**

## Setup
1. Go to **Runtime → Change runtime type → GPU (T4 recommended)**
2. Run All cells (**Runtime → Run all**)
3. Copy the public URL from **Cell 4** into your local `.env` as `TUNNEL_URL`

> **Note:** Colab free tier provides ~12h GPU sessions. Pro/Pro+ gets 24h+.
> Idle disconnect happens after ~90 minutes — the keep-alive cell mitigates this.


In [ ]:
# ─── Configuration ──────────────────────────────────────────
MODEL_NAME   = "qwen2.5-coder:32b"  # Adjust to your GPU VRAM
OLLAMA_PORT  = 11434
TUNNEL_TOOL  = "cloudflare"          # 'cloudflare' recommended
NGROK_TOKEN  = ""                    # Optional fallback
ZROK_TOKEN   = ""                    # Optional fallback
# ────────────────────────────────────────────────────────────

# Detect GPU
import subprocess
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu_info.stdout[:400] if gpu_info.returncode == 0 else "No GPU detected")

In [ ]:
# Cell 1 — Install Ollama
import subprocess, os

print("📦 Installing Ollama on Colab...")
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Ollama install failed:\n{result.stderr[-300:]}")
print("✅ Ollama installed")
v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print("Version:", v.stdout.strip())

In [ ]:
# Cell 2 — Start Ollama Server
import subprocess, time, requests, os

env = os.environ.copy()
env["OLLAMA_HOST"] = f"0.0.0.0:{OLLAMA_PORT}"
env["OLLAMA_ORIGINS"] = "*"

server_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"🚀 Ollama server starting (PID {server_proc.pid})...")

for i in range(30):
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{OLLAMA_PORT}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"✅ Ready after {i+1}s")
            break
    except Exception:
        pass
else:
    raise RuntimeError("Server failed to start")

In [ ]:
# Cell 3 — Pull Model
import subprocess

print(f"📥 Pulling {MODEL_NAME} (may take 10–30 min)...")
result = subprocess.run(["ollama", "pull", MODEL_NAME])
if result.returncode != 0:
    raise RuntimeError(f"Pull failed for {MODEL_NAME}")

print(f"✅ {MODEL_NAME} loaded")
import requests, json
test = requests.post(
    f"http://localhost:{OLLAMA_PORT}/api/generate",
    json={"model": MODEL_NAME, "prompt": "Say 'READY' only.", "stream": False},
    timeout=60,
)
resp = json.loads(test.text).get("response", "").strip()
print("Inference test:", resp)

In [ ]:
# Cell 4 — Start Reverse Tunnel
import subprocess, threading, time, re

tunnel_url = None

if TUNNEL_TOOL == "cloudflare":
    print("🌐 Starting Cloudflare trycloudflare tunnel...")
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && "
        "chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True
    )
    url_event = threading.Event()
    captured = []

    def _run_cf():
        global tunnel_url
        p = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{OLLAMA_PORT}"],
            stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
        )
        for line in p.stderr:
            captured.append(line)
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if m:
                tunnel_url = m.group(0)
                url_event.set()

    threading.Thread(target=_run_cf, daemon=True).start()
    if not url_event.wait(timeout=30):
        print("Last output:", "\n".join(captured[-5:]))
        raise RuntimeError("Tunnel URL not found")

elif TUNNEL_TOOL == "ngrok":
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    t = ngrok.connect(OLLAMA_PORT, "http")
    tunnel_url = t.public_url

# Output
print("\n" + "="*60)
print(f"  ✅ COLAB TUNNEL ACTIVE")
print(f"  PUBLIC URL : {tunnel_url}")
print(f"  MODEL      : {MODEL_NAME}")
print("="*60)
print(f"\n👉 Add to your local .env:")
print(f"TUNNEL_URL={tunnel_url}")
print(f"LLM_MODEL={MODEL_NAME}")

In [ ]:
# Cell 5 — Colab Keep-Alive + Anti-Disconnect
# Colab disconnects idle kernels after ~90 min.
# This cell sends repeated requests to prevent that.
import time, requests

# Also inject a tiny JavaScript keep-alive in the browser (Colab-specific trick)
try:
    from IPython.display import display, Javascript
    display(Javascript("""
    function keepAlive() {
        document.querySelector('#top-toolbar button.run-button')?.click();
    }
    // Click Run periodically — helps prevent disconnect
    setInterval(() => console.log('LegacyNode keep-alive ping'), 60000);
    """))
except Exception:
    pass

print("💓 Colab keep-alive started (Ctrl+C to stop)")
while True:
    try:
        r = requests.get(f"http://localhost:{OLLAMA_PORT}/api/tags", timeout=5)
        ts = time.strftime("%H:%M:%S")
        status = "✅" if r.status_code == 200 else f"⚠️ {r.status_code}"
        print(f"  [{ts}] Ollama: {status} | URL: {tunnel_url}")
    except Exception as e:
        print(f"  Heartbeat err: {e}")
    time.sleep(60)